# De l'ACP aux auto-encodeurs profonds avec Keras

Un auto-encodeur comprime son entrée en un code de petite taille (l'encodeur), puis la reconstruit depuis ce code (le décodeur) : sa cible est son entrée elle-même. Ce notebook en entraîne plusieurs sur MNIST (60 000 images d'entraînement et 10 000 de test, des chiffres manuscrits 28 × 28 en niveaux de gris), du plus simple au plus profond :

1. Un auto-encodeur linéaire
2. L'auto-encodeur linéaire face à l'ACP
3. Un auto-encodeur profond
4. L'espace latent
5. La détection d'anomalies
6. La taille du code
7. Pour aller plus loin : un auto-encodeur épars

Chaque partie réutilise les modèles des précédentes ; la dernière est facultative. Pas besoin de GPU : le notebook tourne en une vingtaine de minutes sur un processeur, dont la moitié pour la partie 6, qui entraîne six auto-encodeurs.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers
from sklearn.decomposition import PCA

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
dim_entree = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde : ni centrés, ni réduits
X_train = X_train.reshape(-1, dim_entree) / 255
X_test = X_test.reshape(-1, dim_entree) / 255
print(X_train.shape, X_test.shape)


def affiche_images(
  *rangees: np.ndarray, titres: list | np.ndarray | None = None
) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rangees[0])
  _, axes = plt.subplots(
    len(rangees), n, figsize=(n * 1.2, len(rangees) * 1.3), squeeze=False
  )
  for rangee, images in zip(axes, rangees):
    for ax, image in zip(rangee, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, titre in zip(axes[0], [] if titres is None else titres):
    ax.set_title(titre)
  plt.show()


affiche_images(X_train[:10], titres=y_train[:10])

## 1. Un auto-encodeur linéaire

Sans fonction d'activation, l'encodeur et le décodeur ne sont que des applications linéaires (plus un biais). Construisez, comme sur la slide « Auto-encodeur en Keras » mais sans activation :

- `encodeur` : une couche `Dense` de `dim_code = 20` unités, sur des entrées de taille `dim_entree` ;
- `decodeur` : une couche `Dense` de `dim_entree` unités, sur des codes de taille `dim_code` ;
- `autoencodeur`, qui enchaîne les deux, compilé avec l'optimiseur Adam et la perte `"mse"`.

La cellule après la solution l'entraîne.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Input(shape=(dim_entree,))` en tête de chaque `keras.Sequential`, qui fixe la taille de ses entrées
- `layers.Dense(n)` sans argument `activation` : une couche linéaire
- `keras.Sequential([encodeur, decodeur], name="autoencodeur")` : un modèle peut servir de couche à un autre, et `encodeur` et `decodeur` restent utilisables seuls
- `autoencodeur.summary()` doit afficher 15 700 paramètres pour l'encodeur (784 × 20 poids + 20 biais) et 16 464 pour le décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dim_code = 20
encodeur = keras.Sequential(
  [keras.Input(shape=(dim_entree,)), layers.Dense(...)], name="encodeur"
)
decodeur = keras.Sequential([..., ...], name="decodeur")
autoencodeur = keras.Sequential([..., ...], name="autoencodeur")
autoencodeur.compile(optimizer=..., loss=...)
autoencodeur.summary()
```

### Solution

In [ ]:
dim_code = 20
encodeur = keras.Sequential(
  [keras.Input(shape=(dim_entree,)), layers.Dense(dim_code)], name="encodeur"
)
decodeur = keras.Sequential(
  [keras.Input(shape=(dim_code,)), layers.Dense(dim_entree)], name="decodeur"
)
autoencodeur = keras.Sequential([encodeur, decodeur], name="autoencodeur")
autoencodeur.compile(optimizer="adam", loss="mse")
autoencodeur.summary()

### Entraînement

La cible est l'entrée : `fit(X_train, X_train)`. La perte de validation se calcule sur les images de test.

In [ ]:
autoencodeur.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
decodeur_lineaire = decodeur  # gardé pour la partie 2
X_test_lineaire = autoencodeur.predict(X_test, verbose=0)
affiche_images(X_test[:10], X_test_lineaire[:10], titres=y_test[:10])

## 2. L'auto-encodeur linéaire face à l'ACP

La slide « Auto-encodeur linéaire et ACP » l'affirme : entraîné avec la MSE, un auto-encodeur linéaire à code de taille k retrouve le sous-espace des k premières composantes de l'ACP, mais pas ses axes. Vérifiez-le :

- ajustez une `PCA` à `dim_code` composantes sur `X_train`, reconstruisez `X_test` avec elle, et comparez l'erreur quadratique moyenne de sa reconstruction à celle de l'auto-encodeur (`X_test_lineaire`, calculé par la cellule précédente) ;
- les 20 directions du décodeur sont les lignes de son noyau `W` (20 × 784) : mesurez, pour chacune, la part de sa norme qui tombe dans le sous-espace de l'ACP ;
- mesurez enfin le plus grand cosinus, en valeur absolue, entre deux directions du décodeur, puis entre deux composantes de l'ACP.

Les deux erreurs sont-elles proches ? Les directions du décodeur sont-elles dans le sous-espace de l'ACP ? Sont-elles orthogonales, comme les composantes ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`Layer.get_weights`](https://keras.io/api/layers/base_layer/#getweights-method)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `acp = PCA(dim_code).fit(X_train)`, puis `acp.inverse_transform(acp.transform(X_test))` : la reconstruction par l'ACP ; l'erreur : `np.mean((X_test - X_test_acp) ** 2)`
- `W = decodeur_lineaire.layers[0].get_weights()[0]` : le noyau de l'unique couche `Dense` du décodeur, de forme (20, 784), une direction par ligne (`get_weights()[1]` est le biais)
- les lignes de `acp.components_` (20 × 784) sont orthonormées : `W @ acp.components_.T` donne les coordonnées de chaque direction dans la base de l'ACP ; la norme de chaque ligne de ce produit (`np.linalg.norm(..., axis=1)`), divisée par celle de la ligne de `W`, est la part cherchée (1 : la direction est entièrement dans le sous-espace)
- les cosinus : normalisez les lignes, `V / np.linalg.norm(V, axis=1, keepdims=True)`, puis `V @ V.T` ; hors de la diagonale (`~np.eye(20, dtype=bool)`), 0 pour deux directions orthogonales

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
acp = PCA(...).fit(...)
X_test_acp = ...
erreur_acp = np.mean((X_test - X_test_acp) ** 2)
erreur_lineaire = ...
print(f"Erreur : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}")

W = decodeur_lineaire.layers[0].get_weights()[0]
part = ...
print(f"Part dans le sous-espace de l'ACP : {part.min():.3f} à {part.max():.3f}")


def cosinus_max(V: np.ndarray) -> float:
  V = ...
  cosinus = ...
  return np.abs(cosinus[~np.eye(len(V), dtype=bool)]).max()


print(f"Plus grand cosinus : décodeur {cosinus_max(W):.2f}, ACP {cosinus_max(...):.2f}")
```

### Solution

In [ ]:
acp = PCA(dim_code).fit(X_train)
X_test_acp = acp.inverse_transform(acp.transform(X_test))
erreur_acp = np.mean((X_test - X_test_acp) ** 2)
erreur_lineaire = np.mean((X_test - X_test_lineaire) ** 2)
print(f"Erreur : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}")

W = decodeur_lineaire.layers[0].get_weights()[0]  # (20, 784)
part = np.linalg.norm(W @ acp.components_.T, axis=1) / np.linalg.norm(W, axis=1)
print(f"Part dans le sous-espace de l'ACP : de {part.min():.3f} à {part.max():.3f}")


def cosinus_max(V: np.ndarray) -> float:
  V = V / np.linalg.norm(V, axis=1, keepdims=True)
  cosinus = V @ V.T
  return np.abs(cosinus[~np.eye(len(V), dtype=bool)]).max()


print(
  f"Plus grand cosinus : décodeur {cosinus_max(W):.2f}, ACP {cosinus_max(acp.components_):.2f}"
)

Les deux erreurs sont presque égales : l'auto-encodeur linéaire fait aussi bien que l'ACP, mais pas mieux, puisque l'ACP donne la meilleure reconstruction linéaire possible à 20 dimensions. Ses 20 directions sont presque entièrement dans le sous-espace de l'ACP (à plus de 95 %) : il a trouvé le même sous-espace, aux imprécisions de l'entraînement près. Mais elles ne sont pas orthogonales (le plus grand cosinus vaut environ 0,2, contre 0 pour l'ACP), ni triées par variance expliquée : rien dans la perte ne l'exige. L'ACP donne cette base en plus, et sans entraînement.

## 3. Un auto-encodeur profond

Ajoutez une couche cachée non linéaire de part et d'autre du code, comme sur la slide « Auto-encodeur en Keras », et laissez ouvert le choix de la sortie : écrivez `construit_autoencodeur(dim_code, activation_sortie, perte)`, qui rend l'auto-encodeur compilé :

- `encodeur` : une `Dense` de 128 unités, puis une de `dim_code` unités, toutes deux avec l'activation ReLU ;
- `decodeur` : une `Dense` de 128 unités (ReLU), puis la couche de sortie, de `dim_entree` unités et d'activation `activation_sortie` (`None` : sortie linéaire) ;
- l'auto-encodeur, qui enchaîne les deux, compilé avec l'optimiseur Adam et la perte `perte`.

La cellule après la solution en entraîne deux, avec le code de 20 nombres de la partie 1 : l'un à sortie linéaire et perte `"mse"`, l'autre à sortie sigmoïde et perte `"binary_crossentropy"`. Elle compare leurs erreurs quadratiques moyennes, la seule mesure commune aux deux pertes, à celles de l'ACP et de l'auto-encodeur linéaire, et donne la plage des pixels que chacun rend. Lequel convient à des pixels dans [0, 1], et pourquoi ?

In [ ]:
def construit_autoencodeur(
  dim_code: int, activation_sortie: str | None, perte: str
) -> keras.Model:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Dense(128, activation="relu")` : une couche cachée ; `layers.Dense(dim_entree, activation=activation_sortie)` : la sortie, linéaire quand `activation_sortie` vaut `None`
- `keras.Sequential([encodeur, decodeur], name="autoencodeur")`, puis `autoencodeur.compile(optimizer="adam", loss=perte)` et `return autoencodeur`
- `construit_autoencodeur(20, "sigmoid", "binary_crossentropy").summary()` doit afficher 103 060 paramètres pour l'encodeur, 103 824 pour le décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_autoencodeur(
  dim_code: int, activation_sortie: str | None, perte: str
) -> keras.Model:
  encodeur = keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(128, activation="relu"),
      ...,
    ],
    name="encodeur",
  )
  decodeur = keras.Sequential(
    [keras.Input(shape=(dim_code,)), ..., layers.Dense(dim_entree, activation=...)],
    name="decodeur",
  )
  autoencodeur = keras.Sequential([encodeur, decodeur], name="autoencodeur")
  autoencodeur.compile(...)
  return autoencodeur
```

### Solution

In [ ]:
def construit_autoencodeur(
  dim_code: int, activation_sortie: str | None, perte: str
) -> keras.Model:
  encodeur = keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(128, activation="relu"),
      layers.Dense(dim_code, activation="relu"),
    ],
    name="encodeur",
  )
  decodeur = keras.Sequential(
    [
      keras.Input(shape=(dim_code,)),
      layers.Dense(128, activation="relu"),
      # None : sortie linéaire ; "sigmoid" : sortie dans ]0, 1[
      layers.Dense(dim_entree, activation=activation_sortie),
    ],
    name="decodeur",
  )
  autoencodeur = keras.Sequential([encodeur, decodeur], name="autoencodeur")
  autoencodeur.compile(optimizer="adam", loss=perte)
  return autoencodeur

### Entraînement et comparaison

Deux auto-encodeurs profonds, 10 epochs chacun : la cellule prend quelques minutes.

In [ ]:
X_test_profond = {}
for activation_sortie, perte in [(None, "mse"), ("sigmoid", "binary_crossentropy")]:
  autoencodeur = construit_autoencodeur(dim_code, activation_sortie, perte)
  autoencodeur.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  X_test_pred = autoencodeur.predict(X_test, verbose=0)
  X_test_profond[perte] = X_test_pred
  print(
    f"Sortie {activation_sortie or 'linéaire'}, perte {perte} : "
    f"erreur {np.mean((X_test - X_test_pred) ** 2):.4f}, "
    f"pixels de {X_test_pred.min():.2f} à {X_test_pred.max():.2f}"
  )
print(f"Erreur : ACP {erreur_acp:.4f}, AE linéaire {erreur_lineaire:.4f}")
# En haut les originaux, puis l'AE linéaire, puis les AE profonds (MSE, puis BCE)
affiche_images(
  X_test[:10],
  X_test_lineaire[:10],
  X_test_profond["mse"][:10],
  X_test_profond["binary_crossentropy"][:10],
  titres=y_test[:10],
)
# La suite garde le dernier, à sortie sigmoïde, et ses deux moitiés
encodeur, decodeur = autoencodeur.layers

Avec le même code de 20 nombres, l'auto-encodeur profond à sortie sigmoïde reconstruit nettement mieux que l'ACP et que l'auto-encodeur linéaire : ses couches non linéaires suivent des surfaces courbes, hors de portée d'un sous-espace. Celui à sortie linéaire fait à peine mieux que l'ACP : il rend des pixels hors de [0, 1], sous 0 et au-dessus de 1, et un fond gris là où l'image est blanche, des valeurs que la sigmoïde exclut par construction. Pour des pixels dans [0, 1], la sortie sigmoïde s'impose ; avec elle, la `binary_crossentropy`, le choix courant, qui traite chaque pixel comme une probabilité, ou la MSE. La suite garde cet auto-encodeur, dans `autoencodeur`, `encodeur` et `decodeur`.

## 4. L'espace latent

L'encodeur seul transforme chaque image de test en son code, un point d'un espace à `dim_code` dimensions : l'espace latent.

In [ ]:
codes = encodeur.predict(X_test, verbose=0)
print(codes.shape)

### Où vivent les codes ?

Le décodeur seul pourrait-il fabriquer de nouveaux chiffres à partir de codes tirés au hasard ? Regardez d'abord où vivent les vrais codes, puis essayez :

- pour chacune des 20 dimensions, affichez le minimum, la moyenne et le maximum des `codes`, et comptez les dimensions qui valent 0 pour toutes les images ;
- décodez 10 codes de bruit blanc, tirés uniformément dans [0, 1] sur chaque dimension, puis 10 codes tirés uniformément entre le minimum et le maximum de chaque dimension, et affichez les deux rangées d'images avec `affiche_images`.

Les codes ressemblent-ils à du bruit uniforme dans [0, 1] ? Les images décodées ressemblent-elles à des chiffres ? Pourquoi ?

**Documentation utile**

- [`numpy.random.Generator.uniform`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.uniform.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `codes.min(axis=0)`, `codes.mean(axis=0)`, `codes.max(axis=0)` : une valeur par dimension ; `np.round(..., 1)` pour les afficher
- une dimension qui vaut 0 pour toutes les images : `codes.max(axis=0) == 0` (la ReLU du code ne rend jamais moins de 0), puis `.sum()`
- `rng = np.random.default_rng(0)`, puis `rng.uniform(0, 1, size=(10, dim_code))` ; `rng.uniform(bas, haut, size=(10, dim_code))` accepte aussi des tableaux `bas` et `haut`, une valeur par dimension
- `decodeur.predict(bruit_blanc, verbose=0)` rend 10 images de 784 pixels, que `affiche_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bas, haut = ..., ...
print("Min :", np.round(bas, 1))
print("Moyenne :", ...)
print("Max :", ...)
print("Dimensions mortes :", int(...))

rng = np.random.default_rng(0)
bruit_blanc = rng.uniform(...)
dans_plage = rng.uniform(...)
affiche_images(
  decodeur.predict(bruit_blanc, verbose=0), decodeur.predict(..., verbose=0)
)
```

#### Solution

In [ ]:
bas, haut = codes.min(axis=0), codes.max(axis=0)
print("Min :", np.round(bas, 1))
print("Moyenne :", np.round(codes.mean(axis=0), 1))
print("Max :", np.round(haut, 1))
print("Dimensions mortes :", int((haut == 0).sum()))

rng = np.random.default_rng(0)
bruit_blanc = rng.uniform(0, 1, size=(10, dim_code))
dans_plage = rng.uniform(bas, haut, size=(10, dim_code))
# En haut le bruit blanc, en bas les codes tirés dans leur plage
affiche_images(
  decodeur.predict(bruit_blanc, verbose=0), decodeur.predict(dans_plage, verbose=0)
)

Les codes ne ressemblent pas à du bruit uniforme dans [0, 1] : la ReLU les rend positifs ou nuls, leur plage change d'une dimension à l'autre et va jusqu'à plusieurs dizaines, et une dimension ou plus peut être morte, nulle pour toutes les images. Le bruit blanc tombe donc dans un coin de l'espace latent, près de l'origine, que le décodeur n'a presque jamais vu : il rend des taches qui se ressemblent toutes. Tirés dans leur plage, les codes ne donnent pas mieux : les vrais codes n'occupent qu'une petite partie de ce pavé, car leurs dimensions sont liées entre elles. Un auto-encodeur compresse, mais ne génère pas : son décodeur ne sait décoder que les codes que produit l'encodeur. L'auto-encodeur variationnel, dans le notebook suivant, impose aux codes une loi connue, $\mathcal{N}(0, I)$, dans laquelle on peut tirer.

### Les centroïdes des chiffres

Les codes ne sont pas dispersés au hasard : ceux d'un même chiffre se regroupent. Le code moyen de chaque chiffre, son centroïde, décodé, en donne une image type.

In [ ]:
centroides = np.stack([codes[y_test == chiffre].mean(axis=0) for chiffre in range(10)])
affiche_images(decodeur.predict(centroides, verbose=0), titres=list(range(10)))

### Interpoler entre deux centroïdes

Les codes régulièrement espacés sur le segment qui relie deux centroïdes, une fois décodés, montrent comment le décodeur passe d'un chiffre à l'autre. Essayez d'autres paires.

In [ ]:
def parcours_latent(debut: int, fin: int, n: int = 12) -> None:
  # n codes régulièrement espacés, du centroïde de debut à celui de fin
  parcours = np.linspace(centroides[debut], centroides[fin], n)
  affiche_images(decodeur.predict(parcours, verbose=0))


parcours_latent(7, 1)
parcours_latent(3, 8)

## 5. La détection d'anomalies

Un auto-encodeur reconstruit bien ce qui ressemble à ses exemples d'entraînement, et mal le reste (slide « Détection d'anomalies par erreur de reconstruction »). Le vôtre n'a vu que des chiffres : la cellule suivante lui donne des anomalies, 1 000 images de vêtements de Fashion-MNIST, dans `X_anomalies`. Le seuil se fixe sur des normaux que le modèle n'a pas appris : les images de test, dont `fit` ne se sert que pour afficher la perte de validation, coupées en deux moitiés, une image sur deux, `X_val` pour fixer le seuil et `X_normal` pour le mettre à l'épreuve (deux moitiés contiguës ne se ressemblent pas assez : les images de MNIST ne sont pas rangées au hasard). La cellule les reconstruit toutes, dans `X_val_pred`, `X_normal_pred` et `X_anomalies_pred`. Puis :

- calculez l'erreur quadratique moyenne de reconstruction de chaque image, dans `erreurs_val`, `erreurs_normales` et `erreurs_anomalies` ;
- fixez le seuil au quantile à 99 % de `erreurs_val`, et donnez la part des normaux de `X_normal`, puis celle des anomalies, qui le dépassent ;
- affichez, au-dessus de leur reconstruction, les 10 images de `X_normal` les plus mal reconstruites, les 10 mieux reconstruites, puis 10 anomalies.

Quelle part des anomalies le seuil détecte-t-il, et quelle part des normaux signale-t-il à tort ? Les normaux les plus mal reconstruits sont-ils des chiffres mal écrits, ou d'une classe précise ? Et les mieux reconstruits ?

**Documentation utile**

- [`numpy.quantile`](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Les normaux : les images de test, que fit n'a pas apprises, une sur deux de chaque côté
X_val, X_normal = X_test[::2], X_test[1::2]
y_normal = y_test[1::2]
# Les anomalies : 1 000 images de Fashion-MNIST, des vêtements
(X_fashion, _), _ = keras.datasets.fashion_mnist.load_data()
X_anomalies = X_fashion[:1000].reshape(-1, dim_entree) / 255
X_val_pred = autoencodeur.predict(X_val, verbose=0)
X_normal_pred = autoencodeur.predict(X_normal, verbose=0)
X_anomalies_pred = autoencodeur.predict(X_anomalies, verbose=0)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.mean((X_val - X_val_pred) ** 2, axis=1)` : une erreur par image (la moyenne sur ses 784 pixels)
- `np.quantile(erreurs_val, 0.99)` : la valeur que dépassent 1 % des erreurs de `X_val` ; `(erreurs_normales > seuil).mean()` : la part des images de `X_normal` au-delà, environ 1 % si elles ressemblent à celles de `X_val`
- `np.argsort(erreurs_normales)` : les indices des images, de la plus petite erreur à la plus grande ; `[::-1]` inverse l'ordre
- `affiche_images(X_normal[pires], X_normal_pred[pires], titres=y_normal[pires])` : les originaux au-dessus de leur reconstruction

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
erreurs_val = ...
erreurs_normales = ...
erreurs_anomalies = ...
seuil = ...
print(f"Normaux au-delà : {...:.1%}, anomalies au-delà : {...:.1%}")

ordre = np.argsort(erreurs_normales)
pires, meilleurs = ordre[::-1][:10], ...
affiche_images(X_normal[pires], X_normal_pred[pires], titres=y_normal[pires])
affiche_images(...)
affiche_images(X_anomalies[:10], ...)
```

### Solution

In [ ]:
erreurs_val = np.mean((X_val - X_val_pred) ** 2, axis=1)
erreurs_normales = np.mean((X_normal - X_normal_pred) ** 2, axis=1)
erreurs_anomalies = np.mean((X_anomalies - X_anomalies_pred) ** 2, axis=1)
seuil = np.quantile(erreurs_val, 0.99)
print(f"Seuil : {seuil:.4f}")
print(f"Normaux au-delà : {(erreurs_normales > seuil).mean():.1%}")
print(f"Anomalies au-delà : {(erreurs_anomalies > seuil).mean():.1%}")

ordre = np.argsort(erreurs_normales)
pires, meilleurs = ordre[::-1][:10], ordre[:10]
print("Les normaux les plus mal reconstruits")
affiche_images(X_normal[pires], X_normal_pred[pires], titres=y_normal[pires])
print("Les mieux reconstruits")
affiche_images(
  X_normal[meilleurs], X_normal_pred[meilleurs], titres=y_normal[meilleurs]
)
print("Des anomalies")
affiche_images(X_anomalies[:10], X_anomalies_pred[:10])

Environ 1 % des normaux de `X_normal` dépassent le seuil, comme prévu : `X_val` et `X_normal` viennent des mêmes données. La grande majorité des anomalies le dépassent, près de 9 sur 10 : l'auto-encodeur n'a jamais vu de vêtement, et les reconstruit en taches qui tirent vers un chiffre. Le seuil ne les attrape pas toutes : en le baissant, on en attraperait davantage, au prix de plus de fausses alertes. Les normaux les plus mal reconstruits sont des chiffres de forme rare, épais ou mal formés : ils s'éloignent de ce que l'auto-encodeur a vu, et sa reconstruction les ramène vers un chiffre plus courant. Les mieux reconstruits sont surtout des 1 : peu de pixels allumés, une forme simple, donc peu d'erreur. L'erreur dépend donc aussi de la classe : avec un seul seuil, un chiffre compliqué est plus souvent signalé qu'un 1.

## 6. La taille du code

Un code plus grand reconstruit mieux, mais compresse moins (slide « Code sous-complet ou sur-complet »). Mesurez ce compromis, avec `construit_autoencodeur` :

- entraînez un auto-encodeur à sortie sigmoïde et perte `"binary_crossentropy"` pour chaque taille de code de 2, 4, 8, 16, 32 et 64, pendant 10 epochs chacun, avec le même `fit` que dans la partie 3 ;
- gardez chaque modèle entraîné dans un dictionnaire `autoencodeurs`, et sa dernière perte de validation dans `pertes_val`, indexés par la taille du code ;
- tracez la perte de validation en fonction de la taille du code, avec un axe des abscisses en échelle logarithmique de base 2 ;
- affichez les 10 premières images de test, puis leurs reconstructions par chaque modèle, une rangée par taille de code.

À partir de quelle taille les chiffres reconstruits deviennent-ils nets ? Quelle taille de code choisiriez-vous, et quel taux de compression (784 / taille du code) obtenez-vous alors ?

**Documentation utile**

- [`Model.fit` (valeur rendue : `History`)](https://keras.io/api/models/model_training_apis/#fit-method)
- [`matplotlib.pyplot.xscale`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xscale.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `construit_autoencodeur(taille, "sigmoid", "binary_crossentropy")` : l'auto-encodeur de la partie 3, pour un code de `taille` nombres
- `historique = autoencodeurs[taille].fit(...)` : `historique.history["val_loss"]` est la liste des pertes de validation, une par epoch, et `[-1]` la dernière
- la boucle prend quelques minutes : `print` la perte de chaque taille au fil de l'eau
- `plt.plot(list(pertes_val), list(pertes_val.values()), marker="o")`, puis `plt.xscale("log", base=2)`
- `affiche_images(X_test[:10], *reconstructions)` : `*` passe chaque élément de la liste `reconstructions`, une reconstruction par modèle, comme une rangée ; `modele(X_test[:10])` appelle le modèle directement, plus vite que `predict` pour 10 images, et `keras.ops.convert_to_numpy` en fait un tableau NumPy

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
autoencodeurs, pertes_val = {}, {}
for taille in [2, 4, 8, 16, 32, 64]:
  autoencodeurs[taille] = construit_autoencodeur(...)
  historique = autoencodeurs[taille].fit(...)
  pertes_val[taille] = ...
  print(f"Code de taille {taille} : {pertes_val[taille]:.4f}")

plt.plot(...)
plt.xscale(...)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

reconstructions = [keras.ops.convert_to_numpy(modele(...)) for modele in ...]
affiche_images(X_test[:10], *reconstructions, titres=y_test[:10])
```

### Solution

In [ ]:
autoencodeurs, pertes_val = {}, {}
for taille in [2, 4, 8, 16, 32, 64]:
  autoencodeurs[taille] = construit_autoencodeur(
    taille, "sigmoid", "binary_crossentropy"
  )
  historique = autoencodeurs[taille].fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  pertes_val[taille] = historique.history["val_loss"][-1]
  print(
    f"Code de taille {taille} (compression {dim_entree / taille:.0f}x) : "
    f"perte de validation {pertes_val[taille]:.4f}"
  )

plt.plot(list(pertes_val), list(pertes_val.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

# En haut les originaux, puis une rangée par taille de code, de 2 à 64
reconstructions = [
  keras.ops.convert_to_numpy(modele(X_test[:10])) for modele in autoencodeurs.values()
]
affiche_images(X_test[:10], *reconstructions, titres=y_test[:10])

La perte baisse vite tant que le code est très petit, puis de moins en moins : chaque doublement du code rapporte moins que le précédent. Les images le montrent : avec 2 ou 4 nombres, les reconstructions sont floues et plusieurs chiffres se confondent, un 4 ou un 5 devenant un 9 ; à partir de 16, elles sont nettes. Un code de 16 à 32 dimensions est un bon compromis, soit un taux de compression de 784 / 32 = 24,5 à 784 / 16 = 49 (en nombre de valeurs : le code et les pixels sont tous des flottants).

### Le code à deux dimensions

Avec un code de 2 nombres, l'espace latent de `autoencodeurs[2]` se dessine : chaque image de test y devient un point, coloré selon son chiffre, une étiquette que l'auto-encodeur n'a jamais vue (slide « Espace latent d'un auto-encodeur »). À côté, les mêmes images projetées sur les deux premières composantes de l'ACP, la réduction de dimension linéaire de la partie 2.

In [ ]:
codes_2d = autoencodeurs[2].layers[0].predict(X_test, verbose=0)
acp_2d = PCA(2).fit(X_train).transform(X_test)
_, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, points, titre in [
  (axes[0], codes_2d, "Auto-encodeur, code de 2 nombres"),
  (axes[1], acp_2d, "ACP, 2 composantes"),
]:
  for chiffre in range(10):
    selectionnes = points[y_test == chiffre]
    ax.scatter(selectionnes[:, 0], selectionnes[:, 1], s=2, alpha=0.5, label=chiffre)
  ax.set_title(titre)
axes[1].legend(markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

Sans jamais voir d'étiquette, l'auto-encodeur isole certains chiffres mieux que l'ACP : les 1, par exemple, y forment une branche à part. Mais la ReLU de son code écrase beaucoup de codes à 0, sur un axe ou sur les deux : les autres chiffres s'entassent près de l'origine, où ils se mêlent plus que dans l'ACP. Les codes laissent aussi de grands vides, où un point pris au hasard ne se décode pas en un chiffre. Un code sans activation, comme celui de la slide « Espace latent d'un auto-encodeur », s'étale mieux.

## 7. Pour aller plus loin : un auto-encodeur épars

Un auto-encodeur épars a en général un code plus large que l'entrée, mais une pénalité sur ses activations n'en laisse que peu actives pour chaque image ; on garde ici 64 unités pour que l'entraînement reste court, la pénalité agit de même. Construisez `autoencodeur_epars` :

- l'auto-encodeur profond, avec un code de 64 unités, suivi d'une couche `PenaliteL1` qui ajoute à la perte une pénalité L1 de coefficient 1e-3 sur ses activations (la somme de leurs valeurs absolues pour chaque image, en moyenne sur les images du lot) ;
- entraînez-le 10 epochs, avec le même `fit` que dans la partie 6.

Comparez-le à l'auto-encodeur à code de 64 unités de la partie 6, `autoencodeurs[64]` : le nombre moyen d'unités actives (> 0) du code par image de test, le nombre d'unités jamais actives, et l'erreur quadratique moyenne de reconstruction. Que coûte la parcimonie ?

**Documentation utile**

- [Les pertes ajoutées par une couche, `add_loss`](https://keras.io/api/layers/base_layer/#addloss-method)
- [Les régularisations des couches Keras](https://keras.io/api/layers/regularizers/)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Dense(64, activation="relu")`, puis `PenaliteL1(1e-3)` : la couche du code, suivie de la pénalité
- `self.add_loss(self.coef * keras.ops.mean(keras.ops.sum(keras.ops.abs(x), axis=1)))` : dans `call`, la pénalité de la couche, somme des valeurs absolues des 64 activations de chaque image (`axis=1`), en moyenne sur les images du lot ; `call` renvoie ensuite `x` tel quel
- `activity_regularizer=keras.regularizers.L1(...)`, l'argument des couches Keras pour une telle pénalité, a changé d'échelle avec Keras 3.14, qui la divise par la taille du lot : le même coefficient ne donne pas le même code selon la version, d'où la couche écrite à la main
- `autoencodeurs[64].layers[0]` : l'encodeur de l'auto-encodeur de la partie 6 (le premier modèle de son `Sequential`)
- `codes = encodeur.predict(X_test, verbose=0)`, puis `(codes > 0).sum(axis=1).mean()` : le nombre moyen d'unités actives par image ; `(codes.max(axis=0) == 0).sum()` : les unités jamais actives

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class PenaliteL1(layers.Layer):
  def __init__(self, coef: float, **kwargs) -> None:
    super().__init__(**kwargs)
    self.coef = coef

  def call(self, x):
    self.add_loss(...)
    return x


encodeur_epars = keras.Sequential(
  [
    keras.Input(shape=(dim_entree,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    PenaliteL1(...),
  ]
)
decodeur_epars = ...
autoencodeur_epars = keras.Sequential([encodeur_epars, decodeur_epars])
autoencodeur_epars.compile(optimizer="adam", loss="binary_crossentropy")
autoencodeur_epars.fit(...)

for nom, modele in [("Sans pénalité", autoencodeurs[64]), ("L1", autoencodeur_epars)]:
  codes = modele.layers[0].predict(X_test, verbose=0)
  erreur = np.mean((X_test - modele.predict(X_test, verbose=0)) ** 2)
  print(f"{nom} : ...")
```

### Solution

In [ ]:
class PenaliteL1(layers.Layer):
  # Renvoie son entrée telle quelle, et ajoute à la perte coef fois la somme des
  # valeurs absolues des activations de chaque image, en moyenne sur le lot
  def __init__(self, coef: float, **kwargs) -> None:
    super().__init__(**kwargs)
    self.coef = coef

  def call(self, x):
    self.add_loss(self.coef * keras.ops.mean(keras.ops.sum(keras.ops.abs(x), axis=1)))
    return x


encodeur_epars = keras.Sequential(
  [
    keras.Input(shape=(dim_entree,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    PenaliteL1(1e-3),
  ],
  name="encodeur_epars",
)
decodeur_epars = keras.Sequential(
  [
    keras.Input(shape=(64,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(dim_entree, activation="sigmoid"),
  ],
  name="decodeur_epars",
)
autoencodeur_epars = keras.Sequential(
  [encodeur_epars, decodeur_epars], name="autoencodeur_epars"
)
autoencodeur_epars.compile(optimizer="adam", loss="binary_crossentropy")
autoencodeur_epars.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=0,
)

for nom, modele in [("Sans pénalité", autoencodeurs[64]), ("L1", autoencodeur_epars)]:
  codes = modele.layers[0].predict(X_test, verbose=0)
  erreur = np.mean((X_test - modele.predict(X_test, verbose=0)) ** 2)
  print(
    f"{nom} : {(codes > 0).sum(axis=1).mean():.1f} unités actives par image sur 64, "
    f"{(codes.max(axis=0) == 0).sum()} jamais actives, erreur {erreur:.4f}"
  )

Avec la pénalité, chaque image n'active plus qu'une vingtaine d'unités du code au lieu d'une cinquantaine, et une trentaine des 64 unités ne s'activent jamais : la pénalité éteint les unités les moins utiles. La parcimonie se paie en reconstruction, dont l'erreur double presque. Le coefficient règle ce compromis : à 1e-4, l'effet est bien plus faible, une cinquantaine d'unités restant actives ; à 1e-2, le code s'éteint presque, une unité active par image en moyenne. Dans un code épars, chaque unité active porte un motif qu'on peut regarder seul : c'est ce qui sert à interpréter les représentations des grands modèles de langage.